In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from scipy.sparse import hstack
from sklearn.naive_bayes import MultinomialNB

In [2]:
df = pd.read_csv('../Phishing_Email.csv')
df.head

<bound method NDFrame.head of        Unnamed: 0                                         Email Text  \
0               0  re : 6 . 1100 , disc : uniformitarianism , re ...   
1               1  the other side of * galicismos * * galicismo *...   
2               2  re : equistar deal tickets are you still avail...   
3               3  \nHello I am your hot lil horny toy.\n    I am...   
4               4  software at incredibly low prices ( 86 % lower...   
...           ...                                                ...   
18645       18646  date a lonely housewife always wanted to date ...   
18646       18647  request submitted : access request for anita ....   
18647       18648  re : important - prc mtg hi dorn & john , as y...   
18648       18649  press clippings - letter on californian utilit...   
18649       18650                                              empty   

           Email Type  
0          Safe Email  
1          Safe Email  
2          Safe Email  
3      Ph

# **Data Preprocessing**

In [3]:
df.isnull().sum()

Unnamed: 0     0
Email Text    16
Email Type     0
dtype: int64

In [4]:
original_df = df
df = df.dropna(subset=['Email Text'])

print(f"Original rows: {len(original_df)}")
print(f"Cleaned rows: {len(df)}")
print(len(original_df)-len(df))

df.isnull().sum()

Original rows: 18650
Cleaned rows: 18634
16


Unnamed: 0    0
Email Text    0
Email Type    0
dtype: int64

In [5]:

print(df['Email Type'].value_counts())

Email Type
Safe Email        11322
Phishing Email     7312
Name: count, dtype: int64


# **Using Naive Bayers**

In [6]:
lable_mapping = {
    'Phishing Email':1,
    'Safe Email':0
}

df['Email Type'] = df['Email Type'].map(lable_mapping)
y = df['Email Type']
y

0        0
1        0
2        0
3        1
4        1
        ..
18645    1
18646    0
18647    0
18648    0
18649    1
Name: Email Type, Length: 18634, dtype: int64

In [7]:
#df["Combined_text"] = df["sender_email"]+' '+df["subject"]+' '+df['body']
vectorizer = TfidfVectorizer(stop_words='english',max_features=5000)
x = vectorizer.fit_transform(df['Email Text'])
x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1342952 stored elements and shape (18634, 5000)>

In [8]:
x_train,x_temp,y_train,y_temp = train_test_split(x,y,test_size=0.3,random_state=42)
x_verif,x_test,y_verif,y_test = train_test_split(x_temp,y_temp,test_size=0.5)

In [9]:
best_verif_score = 0
best_model = None
best_alpha = 0

alphas_to_test = [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]

for curr_alpha in alphas_to_test:
    model = MultinomialNB(alpha=curr_alpha)
    model.fit(x_train,y_train)
    
    y_verif_pred = model.predict(x_verif)
    curr_score = accuracy_score(y_verif,y_verif_pred)
    
    print("*"*30)
    print(f"Aplha Setting: {curr_alpha} verificatuon score: {curr_score*100:.2f}")
    
    
    if curr_score > best_verif_score:
        best_verif_score = curr_score
        best_model = model
        best_setting = curr_alpha
        
print(f"\n\nBest Alpha setting: {best_setting} best verification score: {best_verif_score}")

******************************
Aplha Setting: 0.01 verificatuon score: 94.53
******************************
Aplha Setting: 0.1 verificatuon score: 94.63
******************************
Aplha Setting: 0.5 verificatuon score: 94.70
******************************
Aplha Setting: 1.0 verificatuon score: 94.74
******************************
Aplha Setting: 5.0 verificatuon score: 94.35
******************************
Aplha Setting: 10.0 verificatuon score: 93.35


Best Alpha setting: 1.0 best verification score: 0.9474060822898033


In [10]:
# 1. Check your dataset balance
print("Dataset Breakdown:")
print(df['Email Type'].value_counts(normalize=True) * 100)

# 2. Look at a detailed report for one of your models
from sklearn.metrics import classification_report
print("\nDetailed Report:")
print(classification_report(y_verif, best_model.predict(x_verif)))

Dataset Breakdown:
Email Type
0    60.759901
1    39.240099
Name: proportion, dtype: float64

Detailed Report:
              precision    recall  f1-score   support

           0       0.94      0.97      0.96      1683
           1       0.96      0.91      0.93      1112

    accuracy                           0.95      2795
   macro avg       0.95      0.94      0.94      2795
weighted avg       0.95      0.95      0.95      2795



In [11]:
y_test_pred = best_model.predict(x_test)
print("=== FINAL TEST RESULTS===")
print(f"Test Accuracy: {accuracy_score(y_test, y_test_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_test_pred))

=== FINAL TEST RESULTS===
Test Accuracy: 95.10%

              precision    recall  f1-score   support

           0       0.94      0.98      0.96      1668
           1       0.97      0.91      0.94      1128

    accuracy                           0.95      2796
   macro avg       0.95      0.94      0.95      2796
weighted avg       0.95      0.95      0.95      2796



# **Input**

In [12]:
def check_email(raw_email,threshold=0.80):
    email = vectorizer.transform([raw_email])
    
    prob = model.predict_proba(email)[0,1]
    
    if prob >= threshold:
        return f'Phishing Email (Confidance: {prob:.2f})'
    else:
        return f'Regular Email (risk: {prob:.2f})'


In [13]:
e_mail = input("Enter your email: ")

print(check_email(e_mail))

Regular Email (risk: 0.39)
